In [1]:
from pyspark.sql import SparkSession

spark=SparkSession.builder.appName("ImagePipeline").master("yarn").config("spark.executor.memory","2g").getOrCreate()

df=spark.read.format("binaryFile").load("/user/root/image-pipeline/input/image_01.jpg")
df.printSchema()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/07 16:23:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/07 16:23:25 WARN Client: Neither spark.yarn.jars nor spark.yarn.archive is set, falling back to uploading libraries under SPARK_HOME.


root
 |-- path: string (nullable = true)
 |-- modificationTime: timestamp (nullable = true)
 |-- length: long (nullable = true)
 |-- content: binary (nullable = true)



In [2]:
df.select("path","length").show(3,truncate=False)

+--------------------------------------------------------------+------+
|path                                                          |length|
+--------------------------------------------------------------+------+
|hdfs://master:9000/user/root/image-pipeline/input/image_01.jpg|17406 |
+--------------------------------------------------------------+------+



In [3]:
all_images = spark.read.format("binaryFile").load("/user/root/image-pipeline/input/*.jpg")

all_images.select("path", "length").show(5, truncate=False)

[Stage 1:>                                                          (0 + 1) / 1]

+--------------------------------------------------------------+------+
|path                                                          |length|
+--------------------------------------------------------------+------+
|hdfs://master:9000/user/root/image-pipeline/input/image_04.jpg|17726 |
|hdfs://master:9000/user/root/image-pipeline/input/image_01.jpg|17406 |
|hdfs://master:9000/user/root/image-pipeline/input/image_03.jpg|16732 |
|hdfs://master:9000/user/root/image-pipeline/input/image_06.jpg|13587 |
|hdfs://master:9000/user/root/image-pipeline/input/image_10.jpg|13281 |
+--------------------------------------------------------------+------+
only showing top 5 rows



In [4]:
!pip config set global.index-url https://mirrors.aliyun.com/pypi/simple/
!pip config set global.trusted-host mirrors.aliyun.com

Writing to /root/.config/pip/pip.conf
Writing to /root/.config/pip/pip.conf


In [5]:
!pip install imagehash Pillow -q

In [6]:
import imagehash, PIL, io
print("安装成功！")

安装成功！


In [7]:
import io
import imagehash
from PIL import Image
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

def compute_phash(binary_data):
    img = Image.open(io.BytesIO(binary_data))
    hash_value = imagehash.phash(img)
    return str(hash_value)

phash_udf = udf(compute_phash, StringType())

In [8]:
# 使用 withColumn 新增一列 'phash'
images_with_hash = all_images.withColumn("phash", phash_udf("content"))

# 显示路径和哈希值
images_with_hash.select("path", "phash").show(5, truncate=False)

[Stage 2:>                                                          (0 + 1) / 1]

+--------------------------------------------------------------+----------------+
|path                                                          |phash           |
+--------------------------------------------------------------+----------------+
|hdfs://master:9000/user/root/image-pipeline/input/image_04.jpg|919955e729d0b90f|
|hdfs://master:9000/user/root/image-pipeline/input/image_01.jpg|acb29d939c954acc|
|hdfs://master:9000/user/root/image-pipeline/input/image_03.jpg|caa7982e906f922f|
|hdfs://master:9000/user/root/image-pipeline/input/image_06.jpg|9d13bd13083ec1fc|
|hdfs://master:9000/user/root/image-pipeline/input/image_10.jpg|d6212d4ef3b6a14c|
+--------------------------------------------------------------+----------------+
only showing top 5 rows



In [9]:
from pyspark.sql.functions import udf, col
from pyspark.sql.types import IntegerType  # 这里改成 IntegerType

def hamming_distance(hash1, hash2):
    return imagehash.hex_to_hash(hash1) - imagehash.hex_to_hash(hash2)

hamming_udf = udf(hamming_distance, IntegerType())

In [10]:
# 我们先只选出我们需要的两列，方便观察
image_hashes = images_with_hash.select("path", "phash")

# 自连接：条件是左表路径 < 右表路径（避免重复配对和与自己配对）
# 然后再筛选出汉明距离小于 10 的图片对
duplicate_pairs = image_hashes.alias("a") \
    .join(image_hashes.alias("b"), 
          col("a.path") < col("b.path"), 
          "inner") \
    .withColumn("distance", hamming_udf(col("a.phash"), col("b.phash"))) \
    .filter(col("distance") <= 10) \
    .select(col("a.path").alias("image_1"), 
            col("b.path").alias("image_2"), 
            col("distance"))

duplicate_pairs.show(truncate=False)

+-------+-------+--------+
|image_1|image_2|distance|
+-------+-------+--------+
+-------+-------+--------+



In [11]:
# 取第一张和最后一张的哈希值
hash1 = images_with_hash.select("phash").first()[0]
hash2 = images_with_hash.select("phash").tail(1)[0][0]
dist = imagehash.hex_to_hash(hash1) - imagehash.hex_to_hash(hash2)
print(f"第一张与最后一张的汉明距离: {dist}")

第一张与最后一张的汉明距离: 28


In [16]:
print("spark:", spark.version)
print("all_images count:", all_images.count())

spark: 3.5.5
all_images count: 14


In [18]:
import cv2
import numpy as np

def compute_sharpness(binary_data):
    nparr = np.frombuffer(binary_data, np.uint8)
    img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # 【你来填】计算拉普拉斯算子并求方差
    variance = cv2.Laplacian(gray, cv2.CV_64F).var()
    
    return float(variance)

In [19]:
from pyspark.sql.functions import udf
from pyspark.sql.types import FloatType

# 确保 compute_sharpness 函数已经在当前环境中定义
sharpness_udf = udf(compute_sharpness, FloatType())

In [20]:
# 使用 withColumn 新增清晰度列
images_with_sharpness = all_images.withColumn("sharpness", sharpness_udf("content"))

# 显示路径和清晰度得分
images_with_sharpness.select("path", "sharpness").show(5, truncate=False)

+--------------------------------------------------------------+---------+
|path                                                          |sharpness|
+--------------------------------------------------------------+---------+
|hdfs://master:9000/user/root/image-pipeline/input/image_04.jpg|2792.321 |
|hdfs://master:9000/user/root/image-pipeline/input/image_01.jpg|3705.9834|
|hdfs://master:9000/user/root/image-pipeline/input/image_03.jpg|3974.8572|
|hdfs://master:9000/user/root/image-pipeline/input/image_06.jpg|1605.1427|
|hdfs://master:9000/user/root/image-pipeline/input/image_10.jpg|1048.8274|
+--------------------------------------------------------------+---------+
only showing top 5 rows



In [21]:
images_with_sharpness.describe("sharpness").show()

+-------+------------------+
|summary|         sharpness|
+-------+------------------+
|  count|                14|
|   mean|1350.0834862845284|
| stddev|1303.7650325356037|
|    min|         46.609543|
|    max|         3974.8572|
+-------+------------------+



In [25]:
sharp_images=images_with_sharpness.filter(images_with_sharpness.sharpness>50)
blurred_images=images_with_sharpness.filter(images_with_sharpness.sharpness<50)

total_count=images_with_sharpness.count()
sharp_count=sharp_images.count()
blurred_count=blurred_images.count()

print(f"总图片数: {total_count}")
print(f"清晰图片数 (sharpness >= {50}): {sharp_count}")
print(f"模糊图片数 (sharpness < {50}): {blurred_count}")

总图片数: 14
清晰图片数 (sharpness >= 50): 13
模糊图片数 (sharpness < 50): 1


In [24]:
blurred_images.select("path", "sharpness").orderBy("sharpness").show(truncate=False)

+--------------------------------------------------------------+---------+
|path                                                          |sharpness|
+--------------------------------------------------------------+---------+
|hdfs://master:9000/user/root/image-pipeline/input/image_08.jpg|46.609543|
|hdfs://master:9000/user/root/image-pipeline/input/image_11.jpg|72.63411 |
|hdfs://master:9000/user/root/image-pipeline/input/image_07.jpg|144.99532|
|hdfs://master:9000/user/root/image-pipeline/input/image_05.jpg|289.8281 |
|hdfs://master:9000/user/root/image-pipeline/input/image_02.jpg|477.62256|
|hdfs://master:9000/user/root/image-pipeline/input/image_12.jpg|746.4488 |
|hdfs://master:9000/user/root/image-pipeline/input/image_14.jpg|921.7755 |
+--------------------------------------------------------------+---------+



In [26]:
# 对 sharp_images（13张清晰图片）计算感知哈希
sharp_with_hash = sharp_images.withColumn("phash", phash_udf("content"))

In [29]:
from pyspark.sql.functions import col

# 选取需要的列
image_hashes = sharp_with_hash.select("path", "phash")

# 自连接：找出所有图片对，且路径不重复、不和自己配对
duplicate_pairs = image_hashes.alias("a").join(image_hashes.alias("b"),col("a.path") < col("b.path"),"inner").withColumn("distance", hamming_udf(col("a.phash"), col("b.phash"))).filter(col("distance") <= 10)

In [30]:
# 显示重复图片对
duplicate_pairs.select(col("a.path").alias("image_1"),
                       col("b.path").alias("image_2"),
                       col("distance")).show(truncate=False)

print(f"重复图片对数: {duplicate_pairs.count()}")

+-------+-------+--------+
|image_1|image_2|distance|
+-------+-------+--------+
+-------+-------+--------+

重复图片对数: 0


In [31]:
all_images_new = spark.read.format("binaryFile") \
    .load("/user/root/image-pipeline/input/*.jpg")

print(f"总图片数: {all_images_new.count()}")

总图片数: 500


In [33]:
spark.stop()